# IIoT Predictive Maintenance & Anomaly Detection — Complete Walkthrough

This is the **complete project in a single notebook** — every step from data
ingestion through predictive maintenance and anomaly detection, each explained in
detail. It's organised into four parts that mirror the natural pipeline stages.

**How to run (Colab):** `Runtime → Run all`. The bootstrap cell below clones the
repo and installs dependencies once; everything after it flows top to bottom.

> Just want a quick, condensed run without the explanations? Use `run_all.ipynb`.

---

### Contents
1. [**Setup & Data**](#part-0) — load & understand AI4I 2020 + NASA C-MAPSS
2. [**EDA & Feature Engineering**](#part-1) — sensor signals, RUL labels, rolling features, sequences
3. [**Predictive Maintenance**](#part-2) — Random Forest + SHAP, LSTM RUL regression
4. [**Anomaly Detection**](#part-3) — Isolation Forest + Autoencoder

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gauravs19/iiot-predictive-maintenance/blob/main/notebooks/predictive_maintenance_tutorial.ipynb)

## Bootstrap the environment (runs once)

Detects Colab and clones + installs if needed, then puts the repo root on the import
path so `from src import ...` works. All four sections below reuse this one setup.

In [ ]:
# --- Environment bootstrap (works locally AND on Google Colab) ---------------
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # On Colab there is no repo yet, so clone it and install dependencies.
    !git clone -q https://github.com/gauravs19/iiot-predictive-maintenance.git
    %cd iiot-predictive-maintenance
    !pip install -q -r requirements.txt

# Make the repo root importable so `from src import ...` works from notebooks/.
def _find_repo_root(start="."):
    p = os.path.abspath(start)
    while p != os.path.dirname(p):
        if os.path.isdir(os.path.join(p, "src")):
            return p
        p = os.path.dirname(p)
    raise RuntimeError("repo root (folder containing src/) not found")

REPO_ROOT = _find_repo_root()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)
print("running on Colab" if IN_COLAB else "running locally")

## Imports (consolidated for the whole notebook)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 40)

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, precision_recall_curve)

from src import data, features, models, utils
print("imports OK")

---

<a id='part-0'></a>
# 🧩 Part 1 of 4 · Setup &amp; Data

*Source section: `00_setup_and_data.ipynb`*

# 00 · Setup & Data

**Goal of this notebook:** get a clean, reproducible starting point. By the end you
will have both open datasets downloaded, validated, and understood at a high level.

This project demonstrates two classic Industrial-IoT / manufacturing ML problems:

| Problem | Question it answers | Dataset | Notebook |
|---|---|---|---|
| **Predictive maintenance** | *Will this machine fail, and how soon?* | AI4I 2020 + NASA C-MAPSS | `02` |
| **Anomaly detection** | *Is this machine behaving abnormally right now?* | NASA C-MAPSS | `03` |

**Why these two datasets?**
- **AI4I 2020** is small, tabular, and *labelled* — perfect for a supervised
  "warm-up": predict a failure flag from a single snapshot of sensor readings.
- **NASA C-MAPSS** is *run-to-failure time-series* data from simulated turbofan
  engines — the canonical benchmark for **Remaining-Useful-Life (RUL)** estimation
  and a realistic setting for **unsupervised** anomaly detection.

Everything downloads at runtime, so nothing large is stored in git.

## Step 3 · Load the AI4I 2020 dataset

`data.load_ai4i()` fetches the dataset straight from the UCI Machine Learning
Repository (via the `ucimlrepo` package, with a CSV-download fallback) and returns
a single tidy `DataFrame`.

**What the data represents:** 10,000 rows, each a snapshot of one synthetic milling
machine. Columns include process parameters (air & process temperature, rotational
speed, torque, tool wear) and a `Machine failure` flag plus five specific
failure-mode flags (tool wear failure, heat dissipation, power, overstrain, random).

We display the shape and the first rows to confirm it loaded correctly.

In [ ]:
ai4i = data.load_ai4i()
print("AI4I shape:", ai4i.shape)
ai4i.head()

### AI4I — data dictionary

Each row is a single snapshot of one synthetic milling machine. The columns:

| Column | Meaning |
|---|---|
| `Type` | product-quality variant — **L** (low, 50%), **M** (medium, 30%), **H** (high, 20%) |
| `Air temperature [K]` | ambient temperature (process input) |
| `Process temperature [K]` | machine process temperature (≈ air temp + 10 K) |
| `Rotational speed [rpm]` | spindle speed (derived from power draw) |
| `Torque [Nm]` | applied torque (≈ 40 Nm on average) |
| `Tool wear [min]` | accumulated tool-wear minutes |
| `Machine failure` | **prediction target** — did the machine fail on this row? |
| `TWF, HDF, PWF, OSF, RNF` | the five specific failure modes (we **drop** these — they leak the label) |

Below we show **one record vertically** so every field is readable, plus summary
statistics for the numeric process parameters.

In [ ]:
print("--- one sample record ---")
display(ai4i.iloc[[0]].T)
print("--- numeric summary ---")
ai4i.describe().T[["mean", "std", "min", "max"]].round(2)

### Inspect the failure balance

Real machines fail *rarely*, and this dataset reflects that — only ~3.4% of rows
are failures. This **class imbalance** is important: it means accuracy alone is a
misleading metric (a model predicting "never fails" would be ~96.6% accurate but
useless). We'll come back to this in notebook `02` by using precision/recall and
class weighting.

In [ ]:
print("Failure rate: %.2f%%" % (100 * ai4i["Machine failure"].mean()))
ai4i["Machine failure"].value_counts()

## Step 4 · Load the NASA C-MAPSS dataset (FD001)

`data.load_cmapss("FD001")` downloads three text files from a community mirror and
returns a dict with three DataFrames:

- **`train`** — 100 engines run from healthy all the way **to failure**. Each row is
  one operational cycle.
- **`test`** — 100 *different* engines, but the series are **truncated** some time
  before failure. The model must estimate how much life remains.
- **`rul`** — the ground-truth remaining cycles for each test engine's final row
  (used only to score predictions).

Each row has: an engine `unit` id, the `cycle` number, 3 operational settings, and
21 sensor measurements (temperatures, pressures, speeds, flow ratios, etc.).

In [ ]:
cmapss = data.load_cmapss("FD001")
for k, v in cmapss.items():
    print(f"{k:6s} shape: {v.shape}")
cmapss["train"].head()

### C-MAPSS — what a single row means

Unlike AI4I, C-MAPSS is **time-series**: each row is **one operational cycle of one
engine**, and an engine's rows in order form its run-to-failure history.

| Column group | Meaning |
|---|---|
| `unit` | engine id (1–100). Same id = one engine's full life, ordered by `cycle`. |
| `cycle` | time step (1, 2, 3, …) until failure (train) or censoring (test). |
| `op_setting_1..3` | operating condition that cycle (altitude, Mach number, throttle). |
| `sensor_1..21` | 21 simulated sensors — temperatures, pressures, fan/core speeds, fuel-air ratios, bleed values. |

The raw file has **no ready-made label** — we *derive* the RUL target ourselves (next
section). One engine cycle shown vertically:

In [ ]:
print("--- one engine-cycle record (unit 1, cycle 1) ---")
display(cmapss["train"].iloc[[0]].T)

### How long does each engine survive?

A quick sanity check: in the training set every engine runs to failure, so the max
cycle per unit is its lifetime. The spread below shows engines fail at very
different ages (≈128 to ≈360 cycles) — exactly the variability that makes RUL
prediction a real problem rather than a fixed schedule.

In [ ]:
lifetimes = cmapss["train"].groupby("unit")["cycle"].max()
print("Engine lifetimes — min: %d  median: %d  max: %d"
      % (lifetimes.min(), lifetimes.median(), lifetimes.max()))
lifetimes.describe().round(1)

## Step 5 · Where this fits the IIoT reference architecture

This notebook is the **data ingestion** layer. Mapping the project onto a typical
Industrial-IoT stack:

```
 Edge sensors ─▶ Ingest/Store ─▶ Feature engineering ─▶ ML models ─▶ Serving/Action
 (turbofan,        (this nb:        (notebook 01)        (nb 02/03)    (future:
  milling)          load + cache)                                      iiot-ai-rag)
```

**Next:** notebook `01` explores the sensors visually and engineers the features
both model families will consume.

---

<a id='part-1'></a>
# 🧩 Part 2 of 4 · EDA &amp; Feature Engineering

*Source section: `01_eda_and_features.ipynb`*

# 01 · Exploratory Data Analysis & Feature Engineering

**Goal:** understand the sensor signals and turn raw readings into features the
models can learn from. Good features matter more than fancy models — this notebook
is where most of the predictive power actually comes from.

We cover:
1. Which C-MAPSS sensors carry a usable degradation signal (and which are dead).
2. Building the **RUL label** for supervised training.
3. **Rolling-window features** that capture *trends*, not just instantaneous values.
4. Reshaping the time-series into **sequences** for the LSTM.

## Step 2 · Load the data again

Because notebooks run independently, we reload C-MAPSS here. The download is cached
on disk from notebook `00`, so this is instant the second time.

In [ ]:
cmapss = data.load_cmapss("FD001")
train = cmapss["train"].copy()
print("train shape:", train.shape)

## Step 3 · Which sensors actually carry information?

C-MAPSS has 21 sensors, but in the FD001 operating regime several are **flat lines**
— they never change, so they can't help predict anything. Feeding constant columns
to a model just adds noise and slows training.

Below we compute the standard deviation of each sensor. Sensors with (near-)zero
variance are "dead". Our `features.feature_columns()` helper drops a known list of
these so every model uses the same clean inputs.

In [ ]:
sensor_cols = [c for c in train.columns if c.startswith("sensor_")]
std = train[sensor_cols].std().sort_values()
print("Lowest-variance (likely dead) sensors:")
print(std.head(8).round(4))
print("\nActive feature columns used downstream:")
print(features.feature_columns(train))

## Step 4 · Visualise sensor degradation over an engine's life

The core intuition behind predictive maintenance: as a machine wears out, its
sensor readings **drift**. Below we plot a few informative sensors for a single
engine against its cycle number. You should see clear upward/downward trends as the
engine approaches failure — that drift is the signal our models exploit.

In [ ]:
unit1 = train[train["unit"] == 1]
show = ["sensor_2", "sensor_3", "sensor_4", "sensor_7", "sensor_11", "sensor_15"]
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for ax, s in zip(axes.ravel(), show):
    ax.plot(unit1["cycle"], unit1[s])
    ax.set_title(s); ax.set_xlabel("cycle")
fig.suptitle("Engine #1 — sensor drift toward failure", y=1.02)
fig.tight_layout(); plt.show()

## Step 5 · Build the RUL (Remaining Useful Life) label

For supervised training we need a target. For each row, **RUL = (engine's last
cycle) − (current cycle)** — i.e. how many cycles remain before failure.

**One subtlety (and why we clip):** early in an engine's life there's no visible
degradation, so a literally-correct RUL of, say, 300 is unlearnable from sensors
that look perfectly healthy. The standard C-MAPSS convention is a **piecewise-linear
RUL capped at ~125**: we treat anything healthier than 125 cycles-to-go as "125+".
This matches the physics (degradation only becomes observable near end-of-life) and
trains far better. `features.add_rul()` does exactly this.

In [ ]:
train = features.add_rul(train, clip=125)
print("RUL range after clipping:", train["rul"].min(), "to", train["rul"].max())

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(train[train.unit == 1]["cycle"], train[train.unit == 1]["rul"])
ax.set_title("Clipped RUL target for engine #1 (flat at 125, then linear to 0)")
ax.set_xlabel("cycle"); ax.set_ylabel("RUL"); plt.show()

## Step 6 · Rolling-window features (capturing *trend*)

A single sensor reading is a snapshot; degradation is about **change over time**. We
add, per engine, a **rolling mean** (smooths noise, shows the trend) and **rolling
standard deviation** (rising variance often precedes failure) over a short window.

`features.add_rolling_features()` computes these *per unit* so one engine's history
never leaks into another's.

In [ ]:
cols = features.feature_columns(train)
train_fe = features.add_rolling_features(train, cols, window=5)
new_cols = [c for c in train_fe.columns if c.endswith(("_rmean", "_rstd"))]
print(f"Added {len(new_cols)} rolling features. Example new columns:")
print(new_cols[:6])
train_fe.filter(regex="sensor_2(_rmean|_rstd)?$").head()

## Step 7 · Reshape into sequences for the LSTM

Tree models (notebook `02`, AI4I) take a flat row of features. But an **LSTM** learns
from *ordered sequences*, so we slide a fixed-length window (here 30 cycles) over each
engine's history. Every window becomes one training example shaped
`(timesteps=30, features)`, labelled with the RUL at the **end** of the window.

`features.make_sequences()` returns a 3-D array `(n_windows, 30, n_features)` — the
exact shape PyTorch's LSTM expects.

In [ ]:
X_seq, y_seq = features.make_sequences(train_fe, cols, seq_len=30, label="rul")
print("Sequence tensor X:", X_seq.shape, "  (windows, timesteps, features)")
print("Label vector   y:", y_seq.shape)
print("Example label (RUL at end of first window):", y_seq[0])

## Step 8 · Recap

We now have:
- A **clean feature set** (dead sensors removed).
- A **clipped RUL label** suitable for regression.
- **Rolling features** capturing degradation trend.
- **Sequence tensors** ready for the LSTM.

**Next:** notebook `02` trains the predictive-maintenance models — a classifier on
AI4I and an LSTM RUL regressor on C-MAPSS — and explains each metric.

---

<a id='part-2'></a>
# 🧩 Part 3 of 4 · Predictive Maintenance

*Source section: `02_predictive_maintenance.ipynb`*

# 02 · Predictive Maintenance

Two complementary models, two flavours of the same business goal — *act before the
machine breaks*:

1. **Failure classification** on **AI4I** — given one snapshot of a machine, will it
   fail? (supervised, tabular, tree model + SHAP explainability)
2. **RUL regression** on **C-MAPSS** — given an engine's recent sensor history, how
   many cycles of life remain? (supervised, sequence model / LSTM)

We explain every metric as we go, because *which* metric you optimise is the most
important modelling decision in PdM.

## Part A — Failure classification (AI4I)

### Step 2 · Prepare features and split

`features.prepare_ai4i()` returns `(X, y)`:
- It **drops leakage columns** — the five individual failure-mode flags (TWF, HDF,
  …) literally encode the answer, so keeping them would be cheating.
- It one-hot-encodes the machine quality `Type` (L/M/H).

We then split into train/test with **stratification** so the rare failures appear in
the same proportion in both halves.

In [ ]:
ai4i = data.load_ai4i()
X, y = features.prepare_ai4i(ai4i)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print("train:", X_train.shape, " test:", X_test.shape)
print("test failure rate: %.2f%%" % (100 * y_test.mean()))

### Step 3 · Train a Random Forest

We start with a **Random Forest** — an ensemble of decision trees. It's a strong,
low-fuss baseline for tabular data: handles non-linear interactions, needs little
tuning, and gives feature importances for free.

**`class_weight="balanced"`** is the key setting here: it tells the model to pay
proportionally more attention to the rare failure class, counteracting the 96/4
imbalance we saw in notebook `00`.

In [ ]:
clf = RandomForestClassifier(
    n_estimators=300, max_depth=None, class_weight="balanced",
    random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)
print("trained Random Forest with", clf.n_estimators, "trees")

### Step 4 · Evaluate — and why accuracy is the wrong headline

We look at three things:

- **Confusion matrix** — counts of true/false positives/negatives. In PdM the
  expensive mistake is a **false negative** (a real failure we missed).
- **Precision / recall / F1** — *recall* on the failure class = "of all real
  failures, what fraction did we catch?"; *precision* = "of all failure alarms, what
  fraction were real?". These matter far more than overall accuracy.
- **ROC-AUC** — threshold-independent ranking quality (1.0 = perfect, 0.5 = random).

In [ ]:
pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:, 1]

print("Confusion matrix [rows=true, cols=pred]:")
print(confusion_matrix(y_test, pred))
print("\n", classification_report(y_test, pred, digits=3))
print("ROC-AUC: %.3f" % roc_auc_score(y_test, proba))

### Step 5 · Which signals drive failures? (feature importance)

A model you can't explain is hard to trust on a factory floor. Random Forest exposes
**impurity-based importances** — how much each feature reduces prediction error
across the trees. This tells maintenance engineers *which* measurements to watch.

In [ ]:
imp = pd.Series(clf.feature_importances_, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
imp.plot.barh(ax=ax); ax.set_title("Random Forest feature importance")
plt.tight_layout(); plt.show()

### Step 6 · Explainability with SHAP (optional but powerful)

Feature *importance* tells you what matters globally; **SHAP** tells you *why a
specific prediction* was made — how each feature pushed this particular machine
toward "fail" or "healthy". This per-prediction explanation is exactly what a
maintenance work-order needs (and what the future `iiot-ai-rag` project will turn
into natural language).

> SHAP can be slow; we sample 300 test rows to keep it quick.

In [ ]:
try:
    import shap
    sample = X_test.sample(min(300, len(X_test)), random_state=0)
    explainer = shap.TreeExplainer(clf)
    sv = explainer.shap_values(sample)
    sv_pos = sv[1] if isinstance(sv, list) else sv  # positive class
    shap.summary_plot(sv_pos, sample, show=True)
except Exception as e:
    print("SHAP skipped:", e)

## Part B — Remaining-Useful-Life regression (C-MAPSS, LSTM)

### Step 7 · Build sequences and standardise

We rebuild the feature pipeline from notebook `01` (RUL label → rolling features →
sequences), then **standardise** the inputs (zero mean, unit variance). Neural nets
train much better on standardised inputs. Crucially we **fit the scaler on training
data only** and reuse it on test data, to avoid leaking test-set statistics.

In [ ]:
cm = data.load_cmapss("FD001")
train_fe = features.add_rolling_features(features.add_rul(cm["train"], clip=125),
                                         features.feature_columns(cm["train"]))
cols = features.feature_columns(cm["train"])

X_seq, y_seq = features.make_sequences(train_fe, cols, seq_len=30)

scaler = utils.Standardizer().fit(X_seq.reshape(-1, X_seq.shape[-1]))
def scale(a):
    return scaler.transform(a.reshape(-1, a.shape[-1])).reshape(a.shape)
X_seq_s = scale(X_seq)
print("Sequence tensor:", X_seq_s.shape)

### Step 8 · Train the LSTM

An **LSTM** (Long Short-Term Memory network) is a recurrent neural net designed to
learn from sequences — it carries a memory across timesteps, so it can pick up on
*how* sensors are trending, not just their latest value. Our `LSTMRegressor`
(see `src/models.py`) is a 2-layer LSTM feeding a small dense head that outputs a
single number: predicted RUL.

We minimise **MSE** (mean squared error). Watch that both train and validation loss
fall and stay close — a big gap would signal overfitting.

> On Colab this uses the GPU automatically. On CPU, ~20 epochs takes a couple of
> minutes. Reduce `epochs` if you just want a quick look.

In [ ]:
model = models.LSTMRegressor(n_features=X_seq_s.shape[-1], hidden=64, layers=2)
history = models.train_lstm(model, X_seq_s, y_seq, epochs=20, batch_size=256)
utils.plot_loss(history, "LSTM training (MSE)"); plt.show()

### Step 9 · Predict RUL on the held-out test engines

The test set gives each engine's history truncated *before* failure; we take the
**last 30-cycle window** of each and predict its RUL, then compare to the ground
truth in `RUL_FD001.txt`.

We report two metrics:
- **RMSE** — average error in cycles (interpretable: "off by ~X cycles").
- **C-MAPSS score** — the official competition metric. It's **asymmetric**: it
  punishes *late* predictions (claiming more life than there is → unplanned failure)
  far more than early ones (conservative → safe). Lower is better. This asymmetry
  encodes the real-world cost: under-maintenance is worse than over-maintenance.

In [ ]:
X_test_seq = features.last_sequence_per_unit(cm["test"], cols, seq_len=30)
y_pred = models.predict_lstm(model, scale(X_test_seq))
y_true = cm["rul"]["rul"].to_numpy().clip(max=125)  # clip truth to match training

print("Test RMSE       : %.2f cycles" % utils.rmse(y_true, y_pred))
print("C-MAPSS score   : %.1f  (lower is better)" % utils.cmapss_score(y_true, y_pred))
utils.plot_rul_scatter(y_true, y_pred); plt.show()

### Step 10 · Reading the results

- Points **near the diagonal** = accurate predictions.
- Points **below** the line (predicted < true) = conservative → safe.
- Points **above** the line (predicted > true) = optimistic → risky; these drive up
  the C-MAPSS score the most.

A typical FD001 LSTM lands around **RMSE ≈ 15–20 cycles** — solid for a compact model
trained in minutes. Tuning (longer windows, more epochs, bidirectional LSTM) pushes
it lower.

**Next:** notebook `03` tackles the harder, more realistic case — detecting problems
**without any failure labels** (unsupervised anomaly detection).

---

<a id='part-3'></a>
# 🧩 Part 4 of 4 · Anomaly Detection

*Source section: `03_anomaly_detection.ipynb`*

# 03 · Anomaly Detection (unsupervised)

In the real world you usually **don't have labelled failures** — failures are rare,
and labelling every sensor reading is impractical. So the realistic question becomes:

> *Given mostly-normal operating data, can we automatically flag readings that look
> abnormal — potential early warnings — without ever being told what a failure looks
> like?*

This is **unsupervised anomaly detection**. We use two complementary techniques and
compare them:

1. **Isolation Forest** — a tree-based method that isolates outliers.
2. **Autoencoder** — a neural net trained to reconstruct *healthy* data; anything it
   reconstructs poorly is anomalous.

We then validate the unsupervised scores against the RUL we *do* have, to show the
anomaly score genuinely rises as engines approach failure.

## Step 2 · Define "healthy" vs "degraded"

We have no failure labels, but we *do* know each training row's RUL. We use that only
to **construct an evaluation set** (not to train the detectors):

- **Healthy** = rows with high RUL (engine early in life).
- **Degraded** = rows with low RUL (engine near failure) — these *should* score as
  anomalous if our detectors work.

The detectors themselves are trained **only on healthy data**, mimicking a real
deployment where you fit on normal operation and watch for deviations.

In [ ]:
cm = data.load_cmapss("FD001")
df = features.add_rolling_features(features.add_rul(cm["train"], clip=125),
                                   features.feature_columns(cm["train"]))
cols = features.feature_columns(cm["train"])

healthy = df[df["rul"] >= 100]   # train detectors on these
degraded = df[df["rul"] <= 20]   # should be flagged as anomalous
print("healthy rows:", len(healthy), " degraded rows:", len(degraded))

scaler = utils.Standardizer().fit(healthy[cols].to_numpy("float32"))
Xh = scaler.transform(healthy[cols].to_numpy("float32"))
Xd = scaler.transform(degraded[cols].to_numpy("float32"))

## Step 3 · Method 1 — Isolation Forest

**Intuition:** an Isolation Forest builds random trees that repeatedly split the data
on random features. Outliers are "easy to isolate" — they get separated from the herd
in only a few splits, so they end up with **short path lengths**. The model turns that
into an anomaly score.

We fit it on healthy data only, then score both healthy and degraded rows. We expect
the degraded rows to score as anomalies far more often.

In [ ]:
iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
iso.fit(Xh)

# decision_function: higher = more normal. We negate so higher = more anomalous.
score_h = -iso.decision_function(Xh)
score_d = -iso.decision_function(Xd)
print("mean anomaly score — healthy: %.3f   degraded: %.3f" % (score_h.mean(), score_d.mean()))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(score_h, bins=40, alpha=0.6, label="healthy", density=True)
ax.hist(score_d, bins=40, alpha=0.6, label="degraded", density=True)
ax.set_title("Isolation Forest anomaly scores"); ax.legend(); plt.show()

## Step 4 · Method 2 — Autoencoder (reconstruction error)

**Intuition:** an **autoencoder** is a neural net that compresses its input to a small
"bottleneck" and then reconstructs it. If we train it *only on healthy data*, it
becomes expert at rebuilding normal patterns — but when shown a degraded reading it
has never learned, it reconstructs it **poorly**. That **reconstruction error** is our
anomaly score.

This is the same encoder idea that will later feed the `iiot-ai-rag` project: the
bottleneck layer is effectively a learned "signature" of a machine's state.

In [ ]:
ae = models.AutoEncoder(n_features=Xh.shape[1], latent=8)
hist = models.train_autoencoder(ae, Xh, epochs=30, batch_size=256)
utils.plot_loss(hist, "Autoencoder reconstruction loss (healthy data)"); plt.show()

In [ ]:
err_h = models.reconstruction_error(ae, Xh)
err_d = models.reconstruction_error(ae, Xd)
print("mean reconstruction error — healthy: %.4f   degraded: %.4f"
      % (err_h.mean(), err_d.mean()))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(err_h, bins=40, alpha=0.6, label="healthy", density=True)
ax.hist(err_d, bins=40, alpha=0.6, label="degraded", density=True)
ax.set_title("Autoencoder reconstruction error"); ax.legend(); plt.show()

## Step 5 · Quantify: how well do the scores separate the two groups?

The histograms look convincing, but let's be rigorous. We treat "degraded" as the
positive class and compute **ROC-AUC** of each anomaly score — i.e. if we pick a
random degraded row and a random healthy row, how often does the detector score the
degraded one higher? **1.0 = perfect separation, 0.5 = useless.**

This is a fair check *because* the detectors never saw the RUL labels — we're using
them only to grade the unsupervised result.

In [ ]:
y = np.r_[np.zeros(len(Xh)), np.ones(len(Xd))]   # 0=healthy, 1=degraded
auc_iso = roc_auc_score(y, np.r_[score_h, score_d])
auc_ae  = roc_auc_score(y, np.r_[err_h, err_d])
print("ROC-AUC  Isolation Forest: %.3f" % auc_iso)
print("ROC-AUC  Autoencoder     : %.3f" % auc_ae)

## Step 6 · Does the anomaly score rise as a real engine ages?

The ultimate test of usefulness: track one engine across its whole life and plot its
anomaly score over time. A good detector's score should stay low while the engine is
healthy and **climb steadily as it nears failure** — turning into an actionable early
warning.

In [ ]:
unit = df[df["unit"] == 1].sort_values("cycle")
Xu = scaler.transform(unit[cols].to_numpy("float32"))
err_u = models.reconstruction_error(ae, Xu)

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(unit["cycle"], err_u, color="crimson", label="anomaly score")
ax1.set_xlabel("cycle"); ax1.set_ylabel("reconstruction error", color="crimson")
ax2 = ax1.twinx()
ax2.plot(unit["cycle"], unit["rul"], color="steelblue", alpha=0.6, label="RUL")
ax2.set_ylabel("RUL (true)", color="steelblue")
ax1.set_title("Engine #1 — anomaly score climbs as RUL falls")
plt.show()

## Step 7 · Setting an alert threshold

To deploy this you turn the continuous score into a yes/no alarm by choosing a
**threshold**. A common, label-free choice is a high percentile of the *healthy*
score distribution (e.g. the 99th percentile) — meaning "alarm when the machine looks
more abnormal than 99% of its normal operation." Tightening or loosening this trades
**false alarms** against **missed detections**.

In [ ]:
thr = np.percentile(err_h, 99)
flagged = (err_d > thr).mean()
print("threshold (99th pct of healthy): %.4f" % thr)
print("share of degraded rows correctly flagged: %.1f%%" % (100 * flagged))
print("false-alarm rate on healthy rows         : %.1f%%" % (100 * (err_h > thr).mean()))

## Step 8 · Wrap-up & where this goes next

**What we built across the project:**

| Notebook | Capability | Technique |
|---|---|---|
| 00 | Data ingestion | UCI + NASA loaders |
| 01 | Feature engineering | RUL labels, rolling stats, sequences |
| 02 | Predictive maintenance | Random Forest + SHAP, LSTM RUL |
| 03 | Anomaly detection | Isolation Forest + Autoencoder |

**The hand-off to `iiot-ai-rag` (the sibling project):** the autoencoder's bottleneck
gives a compact **vector signature** of each machine state, and notebook `02`'s model
produces failure probabilities / RUL. Those outputs are exactly what a
Retrieval-Augmented-Generation layer will consume — embedding signatures into a vector
DB to retrieve similar past incidents, and letting an LLM write the maintenance
work-order. That's deliberately kept as a **separate project** so the ML core stays
clean and self-contained.